# 02 — Feature Engineering

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

In [34]:
CLEAN_PATH = Path("../data/clean/clean_game_nba.csv")
TEAM_GAMES_PATH = Path("../data/modelling/team_games.csv")
MODEL_DATA_PATH = Path("../data/modelling/model_data.csv")

games = pd.read_csv(CLEAN_PATH)
games['game_date'] = pd.to_datetime(games['game_date'], errors='coerce')

games.head()

,season_id,team_id,team_abbreviation,team_name,game_id,game_date,matchup,wl,min,fgm,...,dreb,reb,ast,stl,blk,tov,pf,pts,plus_minus,video_available
0,22025,1610612744,GSW,Golden State Warriors,22500002,2025-10-21,GSW @ LAL,W,240,38,...,31,40,29,10,4,19,27,119,10,1
1,22025,1610612747,LAL,Los Angeles Lakers,22500002,2025-10-21,LAL vs. GSW,L,240,42,...,32,39,23,7,2,20,21,109,-10,1
2,22025,1610612745,HOU,Houston Rockets,22500001,2025-10-21,HOU @ OKC,L,290,43,...,36,52,23,6,5,25,26,124,-1,1
3,22025,1610612760,OKC,Oklahoma City Thunder,22500001,2025-10-21,OKC vs. HOU,W,290,46,...,27,38,29,12,4,12,27,125,1,1
4,22025,1610612738,BOS,Boston Celtics,22500083,2025-10-22,BOS vs. PHI,L,240,41,...,32,42,16,7,4,11,28,116,-1,1


In [35]:
# home if "vs." appears, away if "@"
games['home_away_flag'] = games['matchup'].apply(
    lambda x: 'home' if 'vs.' in x.lower() else 'away'
)

# win flag: wl column is W/L
games['team_win'] = games['wl'].apply(lambda x: 1 if x == 'W' else 0)

In [36]:
# Drop columns we don't need
drop_cols = ["wl", "matchup", "team_city", "team_abbreviation"]

team_games = games.drop(columns=drop_cols, errors='ignore')

# Sort chronologically by team
team_games = team_games.sort_values(["team_id", "game_date"]).reset_index(drop=True)

team_games.head()

,season_id,team_id,team_name,game_id,game_date,min,fgm,fga,fg_pct,fg3m,...,ast,stl,blk,tov,pf,pts,plus_minus,video_available,home_away_flag,team_win
0,22025,1610612737,Atlanta Hawks,22500082,2025-10-22,240,38,90,0.422,10,...,25,7,6,16,24,118,-20,1,home,0
1,22025,1610612737,Atlanta Hawks,22500090,2025-10-24,240,41,85,0.482,8,...,26,9,5,16,21,111,4,1,away,1
2,22025,1610612737,Atlanta Hawks,22500101,2025-10-25,240,35,85,0.412,16,...,25,7,5,17,23,100,-17,1,home,0
3,22025,1610612737,Atlanta Hawks,22500115,2025-10-27,240,49,98,0.500,11,...,32,8,1,6,17,123,-5,1,away,0
4,22025,1610612737,Atlanta Hawks,22500130,2025-10-29,240,44,93,0.473,13,...,33,11,8,8,26,117,5,1,away,1


In [37]:
numeric_cols = [
    c for c in team_games.columns
    if c not in ["team_id", "team_name", "game_id", "season_id",
                 "game_date", "home_away_flag", "team_win"]
    and pd.api.types.is_numeric_dtype(team_games[c])
]

windows = [3, 5, 10]

for col in numeric_cols:
    for w in windows:
        team_games[f"{col}_roll_{w}"] = (
            team_games.groupby("team_id")[col]
            .rolling(w).mean().shift(1)
            .reset_index(level=0, drop=True)
        )


In [38]:
# streaks
for w in [1, 3, 5, 10]:
    team_games[f"win_streak_{w}"] = (
        team_games.groupby("team_id")["team_win"]
        .rolling(w).sum().shift(1)
        .reset_index(level=0, drop=True)
    )

# rest metrics
team_games["prev_game_date"] = team_games.groupby("team_id")["game_date"].shift(1)
team_games["rest_days"] = (team_games["game_date"] - team_games["prev_game_date"]).dt.days

team_games["is_b2b"] = (team_games["rest_days"] <= 1).astype(int)
team_games["rest_3plus"] = (team_games["rest_days"] >= 3).astype(int)

In [39]:
TEAM_GAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
team_games.to_csv(TEAM_GAMES_PATH, index=False)

print("Saved:", TEAM_GAMES_PATH)

Saved: ..\data\modelling\team_games.csv


In [40]:
home_rows = team_games[team_games['home_away_flag'] == 'home'].copy()
away_rows = team_games[team_games['home_away_flag'] == 'away'].copy()

home_rows = home_rows.add_prefix("home_")
away_rows = away_rows.add_prefix("away_")

In [41]:
model_data = pd.merge(
    home_rows,
    away_rows,
    left_on="home_game_id",
    right_on="away_game_id",
    how="inner"
)

In [42]:
# Create canonical label: home_win (copy from merged home_team_win)
if "home_team_win" in model_data.columns:
    model_data["home_win"] = model_data["home_team_win"].astype(int)
else:
    raise KeyError("Expected column 'home_team_win' not found in model_data.")

In [43]:
for col in numeric_cols:
    h = "home_" + col
    a = "away_" + col
    if h in model_data.columns and a in model_data.columns:
        model_data[f"diff_{col}"] = (
            model_data[h].astype(float) - model_data[a].astype(float)
        )

In [44]:
model_data = model_data.fillna(0)

MODEL_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
model_data.to_csv(MODEL_DATA_PATH, index=False)

print("Saved:", MODEL_DATA_PATH)
print("Shape:", model_data.shape)

Saved: ..\data\modelling\model_data.csv
Shape: (975, 220)
